# Transformer Tutorial

**Course:** AI+Quantum 2026  
**Topics:** Position encoding, attention mechanism, sparse computation, residual connection, mixture of experts

---

Complete the following 5 exercises in order. Each exercise includes task description, implementation requirements, and assessment points.


---

## Position Encoding Tutorial: RoPE (Rotary Position Embedding)

In mainstream large language models (e.g., Qwen, Llama series), relative position encoding has been largely unified by **RoPE (Rotary Position Embedding)**.

---

### Core Idea of RoPE

**By rotating token vectors in the complex plane, the dot product of two token vectors naturally encodes their relative distance.** RoPE injects absolute position at the input, but exhibits relative position properties when computing Attention.

**2D intuition:** For a 2D vector $[q_1, q_2]$, the rotation at position $m$ is:

$$f(q, m) = \begin{pmatrix} \cos(m\theta) & -\sin(m\theta) \\ \sin(m\theta) & \cos(m\theta) \end{pmatrix} \begin{pmatrix} q_1 \\ q_2 \end{pmatrix} = R_m \cdot q$$

---

### Why is RoPE Relative? (Key Derivation)

Attention computes the dot product of $q$ and $k$. RoPE applies $R_m$ to the Query at position $m$ and $R_n$ to the Key at position $n$, yielding:

$$\text{score}(m, n) = (R_m q)^T (R_n k) = q^T \underbrace{R_m^T R_n}_{\text{key}} k$$

**Rotation matrix property:** $R(\alpha)^T = R(-\alpha)$, and $R(\alpha) R(\beta) = R(\alpha + \beta)$. Thus:

$$R_m^T R_n = R(-m\theta) \cdot R(n\theta) = R\big((n-m)\theta\big) = R_{n-m}$$

Substituting: $\text{score}(m, n) = q^T R_{n-m} k$. **Absolute positions $m$, $n$ vanish—only the relative distance $(n-m)$ remains!**

> In Attention, $m$ is the Query position and $n$ is the Key position. The relative distance $n-m$ means "how many steps the Key is ahead of the Query." The model learns "how far apart" semantics rather than "at which absolute slot," which is the advantage of relative position encoding.

---

### Numerical Example: Same Relative Distance → Same Attention Score

Let $q = [1, 0]$, $k = [0.6, 0.8]$, $\theta = \pi/4$. In the table below, $(m,n)=(1,2)$ and $(m,n)=(3,4)$ both have relative distance $n-m=1$:

| (m, n) | Relative distance n-m | Computation | Score |
|--------|----------------------|-------------|-------|
| (1, 2) | 1 | $q^T R_1 k$ | Same |
| (3, 4) | 1 | $q^T R_1 k$ | Same |
| (0, 2) | 2 | $q^T R_2 k$ | Different |

You may verify with code: pairs $(m,n)$ with the same relative distance $(n-m)$ yield the same Attention score.

---

**Higher dimensions:** Pair dimensions and use different frequencies $\theta_i = 10000^{-2i/d}$:

$$R_{\Theta, m} = \text{diag}\left( R_m(\theta_0), R_m(\theta_1), ..., R_m(\theta_{d/2-1}) \right)$$

**Engineering implementation (Hadamard product optimization):**
1. Generate $\cos$ and $\sin$ caches
2. $\text{rotate\_half}(q) = [-q_2, q_1, -q_4, q_3, ...]$
3. $q_{\text{rotated}} = (q \odot \cos(m\theta)) + (\text{rotate\_half}(q) \odot \sin(m\theta))$


## Exercise 1: RoPE Position Encoding Implementation

**Task:** Implement RoPE position encoding as used in mainstream large models, based on the tutorial above.

**Requirements:** Implement `rotate_half`, `get_rotary_cos_sin`, and `apply_rotary_pos_emb` to apply rotary position encoding to Q/K (using Hadamard product + `rotate_half` optimization).

**Assessment:** Engineering implementation of RoPE's multiplicative injection (rotation).


In [ ]:
# Ex 1 hint: position indices & element-wise ops
# positions = torch.arange(seq_len, device=x.device)
# q_rot = q * cos_emb - rotate_half(q) * sin_emb  # Hadamard product

---

## Attention Tutorial: Self-Attention

After RoPE, we need to understand the core computation unit of the Transformer: **Self-Attention**. Exercise 2 asks you to implement Multi-Head Attention, whose foundation is single-head Self-Attention.

---

### Core Idea

**Self-Attention lets each position in the sequence "see" and aggregate information from other positions.** Unlike CNN's local receptive field, Self-Attention achieves global dependency modeling within a single layer.

---

### Meaning of Q, K, V

For the same input $X \in \mathbb{R}^{L \times D}$, three linear transforms yield:

- **Query (Q):** What the current position is "looking for"
- **Key (K):** What each position "can provide"
- **Value (V):** What each position "actually contributes"

$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

> It is called **Self**-Attention because Q, K, V all come from the same input $X$—the sequence attends to itself.

---

### Scaled Dot-Product Attention

**Step 1:** Compute attention scores (dot product of Query and Key)

$$\text{score}_{i,j} = q_i^T k_j$$

**Step 2:** Scale and Softmax normalize (divide by $\sqrt{d_k}$ to prevent gradient vanishing from large dot products)

$$\alpha_{i,j} = \text{softmax}_j\left(\frac{q_i^T k_j}{\sqrt{d_k}}\right)$$

**Step 3:** Weighted sum of Values to get output

$$\text{out}_i = \sum_j \alpha_{i,j} \cdot v_j$$

**Matrix form:**

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

---

### Intuition

| Step | Meaning |
|------|--------|
| $QK^T$ | Similarity (dot product) between Query at position $i$ and Keys at all positions |
| $\text{softmax}(\cdot)$ | Weight distribution over "which positions to attend to" |
| Multiply by $V$ | Aggregate Values from each position by weight to get output at current position |

---

### Relation to Multi-Head Attention

**Multi-Head Attention** runs the above process in parallel across $H$ heads, each with different $W_Q, W_K, W_V$, then concatenates and projects:

$$\text{MHA}(X) = \text{Concat}(\text{head}_1, ..., \text{head}_H) W_O$$

Each head independently computes $\text{Attention}(Q_h, K_h, V_h)$, capturing different dependencies from different subspaces.


## Exercise 2: Implement Multi-Head Attention (MHA)

**Task:** Complete the core component of the Transformer.

**Requirements:** Implement linear projection for $Q$, $K$, $V$; split heads; scaled dot-product attention; and final linear output projection with concatenation.

**Assessment:** Dimension transformation logic inside the Transformer (Batch, Seq_len, Heads, D_k).

**Dimension flow:**
- Input: (B, L, D) → linear projection → (B, L, H×D_k)
- Split heads: (B, L, H×D_k) → (B, H, L, D_k)
- Attention: $\text{softmax}(QK^T/\sqrt{d_k})V$ → (B, H, L, D_k)
- Concatenate: (B, H, L, D_k) → (B, L, H×D_k)
- Output projection: (B, L, H×D_k) → (B, L, D)


In [ ]:
# Ex 2 hint: reshape for multi-head
# q = q.view(B, L, num_heads, head_dim).transpose(1, 2)  # (B, H, L, D_k)
# scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(head_dim)

## Exercise 3: Implement Sparse Attention

**Task:** Improve upon traditional Attention's redundancy on long sequences.

**Requirements:** Replace full Softmax with **Top-K Sparse Attention**. After computing scores, keep only the top $K$ largest values; set the rest to $-\infty$ (so they become 0 after Softmax).

**Assessment:** Concept of sparse computation and flexible modification of the Attention scoring function.

**Hint:** Use `torch.topk` or `torch.scatter`.


In [ ]:
# Ex 3 hint: topk returns values and indices
# topk_vals, topk_idx = scores.topk(K, dim=-1)
# Build sparse_scores: keep top-K, set rest to -inf before softmax

## Exercise 4: Understanding Autoregressive Probability with Transformer

Consider a binary sequence $x_1, x_2, x_3, x_4 \in \{0, 1\}$. There are $2^4 = 16$ possible sequences of length 4.

**This exercise does not require training.** Instead, use a **randomly initialized, untrained** causal Transformer to directly understand how autoregressive models define joint probability distributions.

---

### Requirements

Assume the model decomposes the joint probability autoregressively:

$$p(x_1, x_2, x_3, x_4) = p(x_1) \, p(x_2 \mid x_1) \, p(x_3 \mid x_1, x_2) \, p(x_4 \mid x_1, x_2, x_3)$$

Each conditional probability is output by the causal Transformer.

---

### Task 4.1: Compute Joint Probability for Each Sequence with a Randomly Initialized Causal Transformer

For any sequence, e.g. **1010**, compute autoregressively:

$$p(1010) = p(x_1=1) \, p(x_2=0 \mid x_1=1) \, p(x_3=1 \mid x_1=1, x_2=0) \, p(x_4=0 \mid x_1=1, x_2=0, x_3=1)$$

**Notes:**
- Keep model parameters randomly initialized—**do not train**;
- Forward pass only;
- At each step, obtain the conditional probability by applying softmax to the Transformer's logits for the current token.

---

### Verification: Sum of Probabilities Over All 16 Sequences Equals 1

Verify that the sum of joint probabilities over all 16 sequences equals 1.

---

### Task 4.2: GHZ State Generation with Transformer

In quantum mechanics, the **GHZ state** (Greenberger–Horne–Zeilinger state) is a maximally entangled state for 3 qubits. When measured in the computational basis, it yields outcomes **000** and **111** with probability 1/2 each, and no other outcomes occur.

**Task:** Use a causal Transformer-based language model (can be small, e.g., 1–2 layers) to generate sequences that approximate this GHZ distribution. The codebook is binary: $\{0, 1\}$.

**Requirements:**
- **Causal constraint:** When the first token is 0, subsequent tokens should have very small probability of being 1 (and vice versa: when the first is 1, subsequent tokens should have very small probability of being 0). This enforces the "all same" structure of 000 and 111.
- **Sampling:** Randomly roll out (sample) 100 sequences of length 3 from the model.
- **Target distribution:** The empirical distribution of the 100 samples should be dominated by **000** and **111** (ideally ~50% each).

**Implementation options:**
- **Option A:** Train the model on a dataset composed of equal numbers of 000 and 111 sequences (no other sequences).
- **Option B:** Add architectural tricks (e.g., special initialization, constrained biases, or hand-crafted weights) so that an untrained or minimally trained model produces the desired distribution.

You **must** implement this using the Transformer framework (causal, autoregressive). Provide code and report the empirical distribution of the 100 sampled sequences (e.g., counts of 000, 111, and others).


In [ ]:
# Ex 4 hint: iterate all 16 sequences & accumulate log-prob
# from itertools import product
# for seq in product([0, 1], repeat=4):
#     log_p = 0
#     for i, token in enumerate(seq):
#         logits = model(context[:i+1])  # causal: only past tokens
#         log_p += F.log_softmax(logits[-1], dim=-1)[token].item()

# Ex 4 GHZ hint: Option A — train on [(0,0,0), (1,1,1)] only; Option B — init output bias
# so that p(x_{i+1}=x_1 | x_1,...,x_i) is high (e.g., bias toward repeating first token).
# Sample: for _ in range(100): seq = []; for i in range(3): logits = model(seq); tok = sample(logits); seq.append(tok)

---

## MoE Tutorial: Mixture of Experts (Minimal)

In the Transformer architecture, MoE typically **replaces the standard feed-forward network (FFN) layer**. Its core mechanism has three parts:

1. **Experts:** Split the single large FFN into multiple smaller FFNs with the same structure but independent parameters.

2. **Router (Gating):** A trainable linear layer. For each input token vector, the Router computes routing probabilities for all experts.

3. **Sparse Activation:** To save compute, only the **Top-K** (usually 1 or 2) experts with highest probability process each token. The final output is the weighted sum of these K experts' outputs (weights are the Router's probabilities).

**Core advantage:** Total model parameters grow (capacity ↑), but per-token activated parameters stay the same (inference cost controlled).

## Exercise 5: Implement Top-1 MoE Layer in Transformer

**Task:** Complete the `forward` method of the `MoELayer` class below. We use **Top-1 routing** only (each token is routed to exactly 1 expert). To simplify, input is 2D: `[num_tokens, d_model]` (batch and sequence length are flattened).

**Requirements:**
- **Router:** Linear layer mapping input to expert logits; apply softmax to get routing probabilities.
- **Experts:** Multiple independent MLPs (same structure); each expert: `Linear(d_model, d_ff) → ReLU → Linear(d_ff, d_model)`.
- **Top-1 routing:** For each token, select the expert with highest probability; output = that expert's output (weighted by its probability, or simply use the selected expert's output).

**Assessment:** Sparse activation logic; parameter capacity vs. per-token compute trade-off.

**Output:** $y = g_e \cdot \text{Expert}_e(x)$ where $e = \arg\max \text{router}(x)$ (Top-1).


In [ ]:
# Ex 5: MoELayer skeleton — complete forward()
import torch
import torch.nn as nn

class MoELayer(nn.Module):
    def __init__(self, d_model, d_ff, num_experts):
        super().__init__()
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.ReLU(),
                nn.Linear(d_ff, d_model)
            ) for _ in range(num_experts)
        ])

    def forward(self, x):
        # x: (num_tokens, d_model)
        # TODO: router logits -> softmax -> top-1 expert index
        # TODO: route each token to its expert, aggregate output
        raise NotImplementedError

In [ ]:
# Ex 5 hint: Top-1 routing
# probs = F.softmax(self.router(x), dim=-1)  # (num_tokens, num_experts)
# expert_idx = probs.argmax(dim=-1)  # (num_tokens,) — which expert per token
# Use a loop over experts, or scatter/gather to route tokens to experts

---

## Summary

After completing these 5 exercises, you will have mastered:
1. **RoPE**: Rotary position encoding used in mainstream models (multiplicative injection, compatible with FlashAttention)
2. **MHA**: Dimension transformations and attention computation at the core of Transformer
3. **Sparse Attention**: Computational optimization for long sequences
4. **Autoregressive Probability**: Understanding autoregressive decomposition of joint probability with an untrained causal Transformer, verifying that the 16 sequence probabilities sum to 1; and generating GHZ-like distributions (000/111) using a causal Transformer via training or architectural tricks
5. **MoE**: Sparse activation and the parameter vs. compute trade-off

You may verify your implementations after completing all exercises.
